In [ ]:
#Task 1 Etivity 3
#Use the following code snippet to download unigrams and bigrams data, and load them into their corresponding dataframes, unigrams_df and bigrams_df
from IPython.core.display import display, HTML
import pandas as pd
import math

!wget https://norvig.com/ngrams/count_1w.txt #unigram data
!wget https://norvig.com/ngrams/count_2w.txt #bigram data

filePath1 = "/content/count_1w.txt"
filePath2 = "/content/count_2w.txt"

unigrams_df = pd.read_csv(filePath1,sep='\t',header=None, names=['unigram','count'])
bigrams_df = pd.read_csv(filePath2,sep='\t',header=None, names=['bigram','count'])

print(unigrams_df.shape, bigrams_df.shape)
print(f'Number of unigrams: {unigrams_df.shape[0]} Number of Bigrams: {bigrams_df.shape[0]}')  # df.size VS df.shape

display(unigrams_df.head(100),bigrams_df.head(100))

# Convert columns to string type to handle potential non-string entries
unigrams_df['unigram'] = unigrams_df['unigram'].astype(str)
bigrams_df['bigram'] = bigrams_df['bigram'].astype(str)

#Build lookup dictionaries and make sure words are normalised
unigram_counts = dict(zip(unigrams_df['unigram'], unigrams_df['count']))
bigram_counts = dict(zip(bigrams_df['bigram'], bigrams_df['count']))


#Return the sentence using bigram chain rule (no add-one smoothing)
#Return 0 if the unigram or bigram is missing
def probability(sentence):
  words = sentence.lower().split()
  #If there are no words at all
  if len(words) == 0:
    return 0.0
  #If a single word, we can return the unigram probability
  if len(words) == 1:
    w = words[0]
    if unigram_counts.get(w, 0) > 0:
      prob = unigram_counts.get(w, 0) / sum(unigram_counts.values())
      print(f'Single word sentence: "{w}"')
      print(f'Unigram count for "{w}" = {unigram_counts[w]}')
      print(f'Total unigram count = {sum(unigram_counts.values())}')
      print(f'Unigram probability for "{w}" = {prob}\n')
      return prob
    else:
      return 0.0

  #Sentence info:
  print(f'Sentence: {words}\n')
  #Use log-prob to avoid overflow
  log_prob = 0.0

  #If there are two or more words
  for i in range(1, len(words)):
    w1 = words[i-1].strip()
    w2 = words[i].strip()
    bigram_key = f"{w1} {w2}"

    #Get the counts
    count_bigram = bigram_counts.get(bigram_key, 0)
    count_unigram = unigram_counts.get(w1, 0)
    print(f"{w1} {w2}")
    print(f' Bigram count for "{bigram_key}" = {count_bigram}')
    print(f' Unigram count for "{w1}" = {count_unigram}')

    #If the word is OOV OR bigram missing, prob is 0
    if count_unigram == 0:
      #word is not in vocabulary
      print(f' "{w1}" is OOV, no unigram is found, probability is 0\n')
      return 0.0
    if count_bigram == 0:
      print(f' Bigram "{bigram_key}" unseen, probability is 0\n')
      #Unseen bigram--no smoothing allowed here
      return 0.0

    #Get the probability for the bigram count
    bigram_prob = count_bigram / count_unigram
    print(f' Bigram probability for "{bigram_key}" = {bigram_prob}\n')

    #Accumulate log prob
    log_prob += math.log(count_bigram / count_unigram)

  #Convert back from log space
  total_prob = math.exp(log_prob)
  print(f'Total probability = {total_prob}\n')
  return total_prob

#TESTING
probA = probability("i love you")
brobB = probability('i hate you')
print(f'probability("i love you")>probability("i hate you") is: {probA>brobB}\n')
probability("OOV1 OOV2") # Test for OOV (Out Of Vocabulary) N-grams

--2025-10-24 09:50:20--  https://norvig.com/ngrams/count_1w.txt
Resolving norvig.com (norvig.com)... 158.106.138.13
Connecting to norvig.com (norvig.com)|158.106.138.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4956241 (4.7M) [text/plain]
Saving to: ‘count_1w.txt.9’

count_1w.txt.9      100%[===================>]   4.73M  9.15MB/s    in 0.5s    

2025-10-24 09:50:21 (9.15 MB/s) - ‘count_1w.txt.9’ saved [4956241/4956241]

--2025-10-24 09:50:21--  https://norvig.com/ngrams/count_2w.txt
Resolving norvig.com (norvig.com)... 158.106.138.13
Connecting to norvig.com (norvig.com)|158.106.138.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5566017 (5.3M) [text/plain]
Saving to: ‘count_2w.txt.9’

count_2w.txt.9      100%[===================>]   5.31M  10.3MB/s    in 0.5s    

2025-10-24 09:50:22 (10.3 MB/s) - ‘count_2w.txt.9’ saved [5566017/5566017]

(333333, 2) (286358, 2)
Number of unigrams: 333333 Number of Bigrams: 286358


,unigram,count
0,the,23135851162
1,of,13151942776
2,and,12997637966
3,to,12136980858
4,a,9081174698
...,...,...
95,like,520585287
96,service,519537222
97,x,508609523
98,than,502609275


,bigram,count
0,0Uplink verified,523545
1,0km to,116103
2,1000s of,939476
3,100s of,539389
4,100th anniversary,158621
...,...,...
95,24th of,327460
96,25th anniversary,261023
97,25th of,397735
98,26th of,271707


Sentence: ['i', 'love', 'you']

i love
 Bigram count for "i love" = 3979312
 Unigram count for "i" = 3086225277
 Bigram probability for "i love" = 0.001289378332053626

love you
 Bigram count for "love you" = 5428714
 Unigram count for "love" = 201063526
 Bigram probability for "love you" = 0.02699999402178991

Total probability = 3.4813207257273315e-05

Sentence: ['i', 'hate', 'you']

i hate
 Bigram count for "i hate" = 876611
 Unigram count for "i" = 3086225277
 Bigram probability for "i hate" = 0.0002840398614232463

hate you
 Bigram count for "hate you" = 504048
 Unigram count for "hate" = 21274675
 Bigram probability for "hate you" = 0.023692394830943365

Total probability = 6.7295845445659965e-06

probability("i love you")>probability("i hate you") is: True

Sentence: ['oov1', 'oov2']

oov1 oov2
 Bigram count for "oov1 oov2" = 0
 Unigram count for "oov1" = 0
 "oov1" is OOV, no unigram is found, probability is 0



0.0

In [ ]:
#Task 1 cont. in new cell
from IPython.core.display import display, HTML
import pandas as pd
import math

!wget https://norvig.com/ngrams/count_1w.txt #unigram data
!wget https://norvig.com/ngrams/count_2w.txt #bigram data

filePath1 = "/content/count_1w.txt"
filePath2 = "/content/count_2w.txt"

unigrams_df = pd.read_csv(filePath1,sep='\t',header=None, names=['unigram','count'])
bigrams_df = pd.read_csv(filePath2,sep='\t',header=None, names=['bigram','count'])

print(unigrams_df.shape, bigrams_df.shape)
print(f'Number of unigrams: {unigrams_df.shape[0]} Number of Bigrams: {bigrams_df.shape[0]}')  # df.size VS df.shape

display(unigrams_df.head(100),bigrams_df.head(100))

# Convert columns to string type to handle potential non-string entries
unigrams_df['unigram'] = unigrams_df['unigram'].astype(str)
bigrams_df['bigram'] = bigrams_df['bigram'].astype(str)

#Build lookup dictionaries and make sure words are normalised
unigram_counts = dict(zip(unigrams_df['unigram'], unigrams_df['count']))
bigram_counts = dict(zip(bigrams_df['bigram'], bigrams_df['count']))

#Function for add-one smoothing
def probability_addone(sentence):
  #Sentence proabability, but with smoothing, unlike previous cell
  words = sentence.lower().split()
  #If there are no words at all
  if len(words) == 0:
    return 0.0

  #Number of unique unigrams
  unique_unigrams = len(unigram_counts)
  print(f"\nSentence = {words}\n")
  #Set log prob to 0
  log_prob = 0.0

  #Same info as before if two or more words
  for i in range(1, len(words)):
    w1 = words[i-1].strip()
    w2 = words[i].strip()
    bigram_key = f"{w1} {w2}"

    #Get counts
    count_bigram = bigram_counts.get(bigram_key, 0)
    count_unigram = unigram_counts.get(w1, 0)

    #Add the add-one smoothing formula
    smooth_prob = (count_bigram + 1) / (count_unigram + unique_unigrams)

    #Print statements for debug checks
    print(f"{w1} {w2}")
    print(f' Bigram count for "{bigram_key}" = {count_bigram}')
    print(f' Unigram count for "{w1}" = {count_unigram}')
    print(f' Smooth probability for "{bigram_key}" = {smooth_prob}\n')

    #Accumulate log probability
    log_prob += math.log(smooth_prob)

  total_prob = math.exp(log_prob)
  print(f'Total smooth probability = {total_prob}\n')
  return total_prob


#TESTING
probA = probability_addone("i love you")
brobB = probability_addone('i hate you')
print(f'probability_addone("i love you")>probability_addone("i hate you") is: {probA>brobB}\n')
probability_addone("OOV1 OOV2") # Test for OOV (Out Of Vocabulary) N-grams

--2025-10-24 09:50:55--  https://norvig.com/ngrams/count_1w.txt
Resolving norvig.com (norvig.com)... 158.106.138.13
Connecting to norvig.com (norvig.com)|158.106.138.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4956241 (4.7M) [text/plain]
Saving to: ‘count_1w.txt.11’

count_1w.txt.11     100%[===================>]   4.73M  9.20MB/s    in 0.5s    

2025-10-24 09:50:56 (9.20 MB/s) - ‘count_1w.txt.11’ saved [4956241/4956241]

--2025-10-24 09:50:56--  https://norvig.com/ngrams/count_2w.txt
Resolving norvig.com (norvig.com)... 158.106.138.13
Connecting to norvig.com (norvig.com)|158.106.138.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5566017 (5.3M) [text/plain]
Saving to: ‘count_2w.txt.11’

count_2w.txt.11     100%[===================>]   5.31M  10.2MB/s    in 0.5s    

2025-10-24 09:50:57 (10.2 MB/s) - ‘count_2w.txt.11’ saved [5566017/5566017]

(333333, 2) (286358, 2)
Number of unigrams: 333333 Number of Bigrams: 286358


,unigram,count
0,the,23135851162
1,of,13151942776
2,and,12997637966
3,to,12136980858
4,a,9081174698
...,...,...
95,like,520585287
96,service,519537222
97,x,508609523
98,than,502609275


,bigram,count
0,0Uplink verified,523545
1,0km to,116103
2,1000s of,939476
3,100s of,539389
4,100th anniversary,158621
...,...,...
95,24th of,327460
96,25th anniversary,261023
97,25th of,397735
98,26th of,271707



Sentence = ['i', 'love', 'you']

i love
 Bigram count for "i love" = 3979312
 Unigram count for "i" = 3086225277
 Smooth probability for "i love" = 0.0012892394100007837

love you
 Bigram count for "love you" = 5428714
 Unigram count for "love" = 201063526
 Smooth probability for "love you" = 0.026955311288917923

Total smooth probability = 3.4751849622512034e-05


Sentence = ['i', 'hate', 'you']

i hate
 Bigram count for "i hate" = 876611
 Unigram count for "i" = 3086225277
 Smooth probability for "i hate" = 0.0002840095106063803

hate you
 Bigram count for "hate you" = 504048
 Unigram count for "hate" = 21274675
 Smooth probability for "hate you" = 0.02332695467934641

Total smooth probability = 6.62507698241839e-06

probability_addone("i love you")>probability_addone("i hate you") is: True


Sentence = ['oov1', 'oov2']

oov1 oov2
 Bigram count for "oov1 oov2" = 0
 Unigram count for "oov1" = 0
 Smooth probability for "oov1 oov2" = 3.000012000048e-06

Total smooth probability = 3.000

3.0000120000479984e-06

In [19]:
#Task 2
import random
from IPython.core.display import display, HTML
import pandas as pd

!wget https://norvig.com/ngrams/count_1w.txt #unigram data
!wget https://norvig.com/ngrams/count_2w.txt #bigram data


filePath1 = "/content/count_1w.txt"
filePath2 = "/content/count_2w.txt"


unigrams_df = pd.read_csv(filePath1,sep='\t',header=None, names=['unigram','count'])
bigrams_df = pd.read_csv(filePath2,sep='\t',header=None, names=['bigram','count'])

#Added max words so it can't get too big
def ShannonVisualization(seed="<S>", max_words=20):

    current_word = seed.lower().strip()
    sentence = [current_word]

    for _ in range(max_words - 1):  # generate up to max_words
        #Find all bigrams that start with current_word
        candidates = bigrams_df[bigrams_df['bigram'].str.startswith(current_word + " ")].copy()

        #If no candidates, stop
        if candidates.empty:
            break

        #Extract next words and their counts
        candidates['next_word'] = candidates['bigram'].apply(lambda b: b.split()[1])
        next_words = candidates['next_word'].tolist()
        counts = candidates['count'].tolist()

        #Compute cumulative prob intervals
        total_count = sum(counts)
        probs = [c / total_count for c in counts]
        cumulative = []
        cumulative_sum = 0.0
        for p in probs:
            # Fix: Append a tuple instead of two arguments
            cumulative.append((cumulative_sum, cumulative_sum + p))
            cumulative_sum += p

        #Pick a random number and find which interval it falls into
        r = random.random()
        chosen_word = None  # Initialize chosen_word
        for i, (low, high) in enumerate(cumulative):
            if low <= r < high:
                chosen_word = next_words[i]
                break

        # If no word was chosen (shouldn't happen with correct probabilities, but as a safeguard)
        if chosen_word is None:
            break

        #Append and move on
        sentence.append(chosen_word)
        current_word = chosen_word

    print("Generated sentence:")
    print(" ".join(sentence))

ShannonVisualization("i")

--2025-10-24 10:04:12--  https://norvig.com/ngrams/count_1w.txt
Resolving norvig.com (norvig.com)... 158.106.138.13
Connecting to norvig.com (norvig.com)|158.106.138.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4956241 (4.7M) [text/plain]
Saving to: ‘count_1w.txt.17’

count_1w.txt.17     100%[===================>]   4.73M  9.13MB/s    in 0.5s    

2025-10-24 10:04:13 (9.13 MB/s) - ‘count_1w.txt.17’ saved [4956241/4956241]

--2025-10-24 10:04:13--  https://norvig.com/ngrams/count_2w.txt
Resolving norvig.com (norvig.com)... 158.106.138.13
Connecting to norvig.com (norvig.com)|158.106.138.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5566017 (5.3M) [text/plain]
Saving to: ‘count_2w.txt.17’

count_2w.txt.17     100%[===================>]   5.31M  10.3MB/s    in 0.5s    

2025-10-24 10:04:14 (10.3 MB/s) - ‘count_2w.txt.17’ saved [5566017/5566017]

Generated sentence:
i have never end time and leisure activities for adequate and 